# Maia Lite 0.3.0 — SFT v2.2 Aya Multilingual Builder

**Objetivo:** combinar o corpus SFT v2 original com exemplos **prontos** em português e espanhol do [Aya Dataset](https://huggingface.co/datasets/CohereLabs/aya_dataset), sem traduzir textos e sem GPU. Preservar integralmente a validação v2 e o material científico de treinamento. Metas de treino: **PT 7.000, EN 15.000, ES 8.000** (30.000).

**Integridade:** apenas `Aya/default/train`; `test` do Aya nunca entra no treinamento. Deduplicação exata por prompt e par; a auditoria de duplicatas semânticas, qualidade e licenças de outras fontes continua necessária. Se não houver exemplos aprovados suficientes, a saída será `INCOMPLETE` — **sem inventar, repetir ou traduzir exemplos**.

**Observação:** o Aya tem licença declarada Apache-2.0, mas o corpus misto mantém condições próprias de cada fonte, inclusive ARC (CC BY-SA 4.0) e artigos científicos.

## 1. Dependências (CPU)
Evita o conflito do Colab com `pandas==2.2.3` e `bigframes` que requer `pyarrow>=23.0.1`.

In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow>=23.0.1" "datasets>=3,<5"

Se o Colab solicitar reinício da sessão, reinicie **uma vez** antes da próxima célula.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, random, re, collections, unicodedata, sys
from datetime import datetime, timezone
from datasets import load_dataset
import datasets, pandas, pyarrow
print('Versões:',datasets.__version__,pandas.__version__,pyarrow.__version__)
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
SRC=ROOT/'datasets/sft_v2'
OUT=ROOT/'datasets/sft_v2_2_aya'
OUT.mkdir(parents=True,exist_ok=True)
TRAIN_IN=SRC/'sft_v2_train.jsonl'
VAL_IN=SRC/'sft_v2_validation.jsonl'
for p in (TRAIN_IN,VAL_IN):
    if not p.is_file(): raise FileNotFoundError(f'Arquivo necessário ausente: {p}')
print('Saída:',OUT)

## 2. Parâmetros reprodutíveis

In [ ]:
SEED=20261118
TARGET={'pt':7000,'en':15000,'es':8000}
MIN_PROMPT=12
MIN_ANSWER=8
MAX_PROMPT=3500
MAX_ANSWER=3500
# Segurança: não alterar a validação v2.
# Aya é usado apenas para complementar o treino.
assert sum(TARGET.values())==30000
RNG=random.Random(SEED)
print(TARGET)

## 3. Funções e leitura do SFT v2

In [ ]:
def load_jsonl(path):
    with open(path,encoding='utf-8') as f:
        for line_no,line in enumerate(f,1):
            if line.strip():
                try: yield json.loads(line)
                except Exception as e: raise ValueError(f'{path}:{line_no}: {e}') from e

def write_jsonl(path, rows):
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False,sort_keys=True)+'\n')

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def norm(s):
    return unicodedata.normalize('NFC',str(s or '')).replace('\r\n','\n').strip()

def key(s):
    return re.sub(r'\s+',' ',norm(s)).casefold()

def lang(s):
    return {'pt':'pt','por':'pt','portuguese':'pt','en':'en','eng':'en','english':'en',
            'es':'es','spa':'es','spanish':'es'}.get(str(s).strip().lower())

train=list(load_jsonl(TRAIN_IN)); validation=list(load_jsonl(VAL_IN))
assert train and validation
for split,rows in [('train',train),('validation',validation)]:
    for i,r in enumerate(rows):
        assert all(k in r for k in ('id','groupId','language','prompt','completion')), (split,i,r)
        assert lang(r['language']) in TARGET,(split,i,r['language'])
assert not ({r['groupId'] for r in train}&{r['groupId'] for r in validation}), 'groupId sobreposto no SFT v2'
assert not ({key(r['prompt']) for r in train}&{key(r['prompt']) for r in validation}), 'prompt sobreposto no SFT v2'
print('Original treino:',len(train),collections.Counter(r['language'] for r in train))
print('Validação preservada:',len(validation))

## 4. Seleção do corpus original
Mantemos **todos** os exemplos científicos de treinamento. Selecionamos os demais por amostragem determinística quando houver excesso de um idioma. Os exemplos não selecionados são exportados para auditoria; nenhum arquivo original é modificado.

In [ ]:
by_lang=collections.defaultdict(list)
for r in train: by_lang[lang(r['language'])].append(r)
selected=[]; excluded=[]
for language in TARGET:
    group=by_lang[language]
    fixed=[r for r in group if r.get('category')=='science_custom']
    flexible=[r for r in group if r.get('category')!='science_custom']
    if len(fixed)>TARGET[language]:
        raise ValueError(f'Exemplos científicos ({language}) excedem a meta; ajuste TARGET')
    random.Random(SEED+sum(map(ord,language))).shuffle(flexible)
    capacity=TARGET[language]-len(fixed)
    selected.extend(fixed+flexible[:capacity]);excluded.extend(flexible[capacity:])
print('Base selecionada:',len(selected),collections.Counter(lang(r['language']) for r in selected))
print('Base excedente preservada em arquivo separado:',len(excluded))

## 5. Baixar Aya Dataset (apenas `train`)
Os campos publicados são `inputs`, `targets`, `language_code`, `annotation_type` e `user_id`. O dataset é carregado integralmente em CPU; a seleção será somente PT/ES. Não usamos o conjunto `test`.

In [ ]:
aya=load_dataset('CohereLabs/aya_dataset', 'default', split='train')
print('Aya train:',len(aya))
print('Colunas:',aya.column_names)
assert {'inputs','targets','language_code'}.issubset(set(aya.column_names))
print('PT/ES disponíveis:',collections.Counter(lang(x['language_code']) for x in aya if lang(x['language_code']) in ('pt','es')))

## 6. Filtragem, duplicações e prevenção de vazamento exato
Filtramos textos vazios, limites de tamanho, repetição excessiva simples e identidade prompt=completion. Também bloqueamos prompts e pares já existentes no treinamento **e na validação**. Essas verificações não substituem auditoria semântica.

In [ ]:
reject=collections.Counter()
seen_prompt={key(r['prompt']) for r in train+validation}
seen_pair={(key(r['prompt']),key(r['completion'])) for r in train+validation}
seen_ids={str(r['id']) for r in train+validation}
aya_candidates={'pt':[],'es':[]}
for index,x in enumerate(aya):
    language=lang(x['language_code'])
    if language not in aya_candidates:continue
    p,a=norm(x['inputs']),norm(x['targets'])
    if not (MIN_PROMPT<=len(p)<=MAX_PROMPT and MIN_ANSWER<=len(a)<=MAX_ANSWER):
        reject['length']+=1;continue
    kp,ka=key(p),key(a)
    if kp==ka:reject['echo']+=1;continue
    if kp in seen_prompt or (kp,ka) in seen_pair:
        reject['duplicate_prompt']+=1;continue
    # filtro heurístico simples de resposta repetida (não implica avaliação factual)
    words=ka.split()
    if len(words)>=30 and len(set(words))/len(words)<0.12:
        reject['repetition']+=1;continue
    rid=f'aya-train-{index:07d}'
    if rid in seen_ids:raise ValueError('Colisão inesperada de ID')
    row={'id':rid,'groupId':f'aya:{index}','language':language,
         'category':'conversation','source':'CohereLabs/aya_dataset',
         'license':'apache-2.0','prompt':p,'completion':a,
         'source_split':'train','source_row_index':index,
         'annotation_type':x.get('annotation_type')}
    aya_candidates[language].append(row)
    seen_prompt.add(kp);seen_pair.add((kp,ka));seen_ids.add(rid)
print('Candidatos Aya:',{k:len(v) for k,v in aya_candidates.items()})
print('Rejeições:',dict(reject))

## 7. Completar as metas com dados Aya
Sem duplicar exemplos e sem traduzir. O déficit remanescente fica explícito.

In [ ]:
combined=list(selected)
aya_selected=[]
for language in ('pt','es'):
    need=max(0,TARGET[language]-sum(lang(r['language'])==language for r in combined))
    pool=aya_candidates[language][:]
    random.Random(SEED+100+sum(map(ord,language))).shuffle(pool)
    chosen=pool[:need]
    combined.extend(chosen);aya_selected.extend(chosen)
    print(language,'necessários:',need,'Aya incorporados:',len(chosen),'faltam:',need-len(chosen))
counts=collections.Counter(lang(r['language']) for r in combined)
shortfall={l:max(0,TARGET[l]-counts[l]) for l in TARGET}
print('Resultado:',len(combined),dict(counts),'Déficit:',shortfall)

## 8. Auditoria estrutural final

In [ ]:
assert len(combined)==len({r['id'] for r in combined}), 'IDs duplicados'
assert not ({r['groupId'] for r in combined}&{r['groupId'] for r in validation}), 'Group leak'
train_prompt={key(r['prompt']) for r in combined}
val_prompt={key(r['prompt']) for r in validation}
assert train_prompt.isdisjoint(val_prompt), 'Prompt leak'
assert len(train_prompt)==len(combined), 'Prompts duplicados no treino'
assert all(lang(r['language']) in TARGET for r in combined)
assert len(aya_selected)==sum(r['source']=='CohereLabs/aya_dataset' for r in combined)
assert all(r.get('source_split')=='train' for r in aya_selected)
print('PASS: IDs, grupos, prompts, splits e idiomas')

## 9. Exportação + manifesto + amostras para revisão

In [ ]:
# Evita sobrescrever os arquivos da v2.
train_out=OUT/'sft_v2_2_train.jsonl'
val_out=OUT/'sft_v2_2_validation.jsonl'
extra_out=OUT/'sft_v2_unselected.jsonl'
review_out=OUT/'human_review_sample.jsonl'
write_jsonl(train_out,combined)
write_jsonl(val_out,validation)
write_jsonl(extra_out,excluded)
review=[]
for l in ('pt','en','es'):
    pool=[r for r in combined if lang(r['language'])==l]
    random.Random(SEED+sum(map(ord,l))).shuffle(pool)
    review+=pool[:min(40,len(pool))]
write_jsonl(review_out,review)
status='READY_FOR_HUMAN_REVIEW' if sum(shortfall.values())==0 else 'INCOMPLETE'
manifest={
 'project':'Maia Lite 0.3.0','pipeline':'SFT v2.2 Aya Multilingual Builder',
 'created_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,'status':status,
 'target_train':TARGET,'actual_train':len(combined),'actual_validation':len(validation),
 'train_by_language':dict(counts), 'shortfall_by_language':shortfall,
 'train_by_category':dict(collections.Counter(r.get('category','unknown') for r in combined)),
 'train_by_source':dict(collections.Counter(r.get('source','unknown') for r in combined)),
 'aya_added':len(aya_selected),'v2_original_excluded':len(excluded),
 'rejections':dict(reject),
 'inputs_sha256':{TRAIN_IN.name:sha256(TRAIN_IN),VAL_IN.name:sha256(VAL_IN)},
 'outputs_sha256':{p.name:sha256(p) for p in [train_out,val_out,extra_out,review_out]},
 'aya_dataset':'CohereLabs/aya_dataset','aya_split':'train','aya_license_declared':'apache-2.0',
 'warnings':['Aya test split not used.',
             'Validation is the exact SFT v2 validation; not independent external test.',
             'Only exact prompt overlap checked; semantic leakage not ruled out.',
             'Mixed corpus licenses remain source-specific; review ARC share-alike and scientific papers.',
             'Human quality and safety audit required before SFT.',
             'Target length filters are character-based, not tokenizer-based.']
}
manifest_path=OUT/'manifest_v2_2.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps({k:manifest[k] for k in ('status','actual_train','actual_validation','train_by_language','shortfall_by_language','aya_added')},indent=2,ensure_ascii=False))
print('Arquivos salvos em:',OUT)

## 10. Antes de treinar

1. Envie o `manifest_v2_2.json` e, se possível, `human_review_sample.jsonl` para análise. O número exato de registros PT/ES do Aya só é conhecido após o download.
2. Verifique os exemplos quanto a qualidade, idioma, conteúdo impróprio, duplicatas semânticas e possíveis sobreposições com benchmarks externos. O conjunto científico de validação continua isolado.
3. A licença **Apache 2.0 do Aya não substitui** as licenças dos demais datasets. O uso/distribuição do corpus misto requer revisão das obrigações de cada fonte.
4. Para o LoRA v2, iniciar do **checkpoint base de 300.000 steps** e manter saída distinta do SFT científico v1. **Não treinar** se `status=INCOMPLETE` sem decidir conscientemente como tratar o déficit.

Nenhuma GPU é necessária neste notebook.